In [2]:
%%writefile /content/drive/MyDrive/hioki_lab/src/voice_synthesis/DDSP_VAE_twostream/dataset.py
from torch.utils.data import Dataset, DataLoader

class RealAudioDataset(Dataset):
    def __init__(self, data_dir, crop_len_sec=2.0):
        self.files = glob.glob(os.path.join(data_dir, "*.pt"))
        self.crop_len_samples = int(crop_len_sec * SAMPLE_RATE)
        self.hop_length = HOP_LENGTH
        # フレーム数換算の長さ (f0, loudness, mel用)
        self.crop_len_frames = self.crop_len_samples // HOP_LENGTH

        # 正規化パラメータ
        self.f_min, self.f_max = 20.0, 1000.0
        self.l_min, self.l_max = -80.0, 0.0

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        data = torch.load(path, weights_only=False)

        # 元データを取り出す
        audio = data['audio']       # [samples]
        mel = data['mel']           # [1, 80, T_frames]
        f0 = data['f0']             # [T_frames, 1]
        loudness = data['loudness'] # [T_frames, 1]

        # --- ★ここが追加: ランダムクロップ処理 ---
        total_frames = f0.shape[0]

        # データが要求サイズより長い場合のみ切り出す
        if total_frames > self.crop_len_frames:
            # ランダムな開始フレームを決める
            max_start = total_frames - self.crop_len_frames
            start_frame = np.random.randint(0, max_start)
            end_frame = start_frame + self.crop_len_frames

            # サンプル単位の開始位置
            start_sample = start_frame * self.hop_length
            end_sample = end_frame * self.hop_length

            # スライス実行
            f0 = f0[start_frame:end_frame]
            loudness = loudness[start_frame:end_frame]
            mel = mel[:, :, start_frame:end_frame]
            audio = audio[start_sample:end_sample]

        # --- 正規化 (以前と同じ) ---
        f0_norm = (f0 - self.f_min) / (self.f_max - self.f_min + 1e-7)
        loud_norm = (loudness - self.l_min) / (self.l_max - self.l_min + 1e-7)
        loud_norm = torch.clamp(loud_norm, 0.0, 1.0)

        return {
            'audio': audio,
            'mel': mel,
            'f0_hz': f0,
            'f0_norm': f0_norm,
            'loudness_norm': loud_norm,
            'inst_name': data['instrument_name']
        }

Overwriting /content/drive/MyDrive/hioki_lab/src/voice_synthesis/DDSP_VAE_twostream/dataset.py
